In [ ]:
import sys
import numpy as np
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLabel, QPushButton, QFrame)
from PyQt6.QtCore import QTimer, Qt
import pyqtgraph as pg
from scipy.linalg import solve_discrete_are

class MRA_W_Tracking(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("RL Control - Seguimiento Completo de Pesos W")
        self.resize(1300, 850)
        
        # --- Parámetros de Octave ---
        self.A = np.array([[1.8980, -0.9048], [1.0, 0.0]])
        self.B = np.array([[1.0], [0.0]])
        self.Q = np.eye(2)
        self.R = 2.0
        self.gamma = 0.999
        self.f_olvido = 0.99
        self.con_error_k = 0.1
        self.N = 4000
        self.n_ls = 7
        
        # --- Cálculo LQR Óptimo y W objetivo ---
        self.K_opt, self.H_opt = self.calcular_lqr_optimo()
        # Relación H -> W (incluyendo el factor 2 para términos cruzados)
        self.W_target = [
            self.H_opt[0,0],       # W1 -> H11
            2 * self.H_opt[0,1],   # W2 -> 2*H12
            2 * self.H_opt[0,2],   # W3 -> 2*H13
            self.H_opt[1,1],       # W4 -> H22
            2 * self.H_opt[1,2],   # W5 -> 2*H23
            self.H_opt[2,2]        # W6 -> H33
        ]
        
        self.reset_variables()
        self.init_ui()
        
        self.timer = QTimer()
        self.timer.timeout.connect(self.step)
        self.timer.start(15)

    def reset_variables(self):
        self.H_init = np.array([[14.0, -2.0, 2.0], [-8.0, 3.0, -1.0], [8.0, -5.0, 4.0]])
        self.K = (-(1.0 / self.H_init[2, 2]) * self.H_init[2, 0:2]).reshape(1, 2)
        self.x = np.array([[5.0], [-4.0]])
        self.W_H = np.zeros((6, 1))
        self.P_n = np.eye(6) * 1000
        self.phi_ls = np.zeros((6, self.n_ls))
        self.phi1_ls = np.zeros((6, self.n_ls))
        self.r_ls = np.zeros((self.n_ls, 1))
        self.iter = 1
        self.pos_history = []
        self.w_history = [[] for _ in range(6)]
        self.h_actual = np.zeros((3, 3))

    def calcular_lqr_optimo(self):
        Ag, Bg = np.sqrt(self.gamma) * self.A, np.sqrt(self.gamma) * self.B
        P = solve_discrete_are(Ag, Bg, self.Q, self.R)
        K = -np.linalg.inv(self.R + self.gamma * self.B.T @ P @ self.B) @ (self.gamma * self.B.T @ P @ self.A)
        H = np.block([[self.Q + self.gamma * self.A.T @ P @ self.A, self.gamma * self.A.T @ P @ self.B],
                      [self.gamma * self.B.T @ P @ self.A, self.R + self.gamma * self.B.T @ P @ self.B]])
        return K.flatten(), H

    def init_ui(self):
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QHBoxLayout(central_widget)

        # Izquierda: Animación y Pesos W
        left_layout = QVBoxLayout()
        
        self.canvas = pg.PlotWidget(title="Masa (x1)")
        self.canvas.showGrid(x=True, y=True)
        self.canvas.setYRange(-10, 10)
        self.rect = pg.ScatterPlotItem(size=20, brush='r')
        self.canvas.addItem(self.rect)
        left_layout.addWidget(self.canvas)

        self.plot_w = pg.PlotWidget(title="Evolución de Pesos W (Target Punteado)")
        self.plot_w.setXRange(0, self.N)
        self.plot_w.addLegend()
        self.w_curves = []
        for i in range(6):
            color = pg.intColor(i)
            # Línea de convergencia real
            self.w_curves.append(self.plot_w.plot(pen=pg.mkPen(color, width=2), name=f"W{i+1}"))
            # Línea objetivo (punteada)
            target_line = pg.InfiniteLine(pos=self.W_target[i], angle=0, 
                                          pen=pg.mkPen(color, width=1, style=Qt.PenStyle.DashLine))
            self.plot_w.addItem(target_line)
        
        left_layout.addWidget(self.plot_w)
        main_layout.addLayout(left_layout, stretch=2)

        # Derecha: Posición y Datos
        right_layout = QVBoxLayout()
        self.plot_res = pg.PlotWidget(title="Respuesta Temporal")
        self.curve_pos = self.plot_res.plot(pen='y')
        right_layout.addWidget(self.plot_res)

        # Panel numérico
        self.lbl_k = QLabel(""); self.lbl_h = QLabel("")
        for lbl in [self.lbl_k, self.lbl_h]: lbl.setStyleSheet("font-family: monospace; font-size: 10px;")
        
        right_layout.addWidget(QLabel("<b>DATOS RL vs OPT</b>"))
        right_layout.addWidget(self.lbl_k)
        right_layout.addWidget(self.lbl_h)

        self.btn_reset = QPushButton("REINICIAR")
        self.btn_reset.clicked.connect(self.reset_variables)
        right_layout.addWidget(self.btn_reset)
        
        self.lbl_iter = QLabel("Iteración: 0")
        right_layout.addWidget(self.lbl_iter)
        main_layout.addLayout(right_layout, stretch=1)

    def step(self):
        if self.iter > self.N: return

        # Dinámica (Misma de Octave)
        noise = 0.20099 * np.exp(-0.000269 * self.iter) * (np.sin(self.iter)**3 * np.cos(self.iter))
        u = float(np.dot(self.K, self.x)) + noise
        x_curr, x_next = self.x, np.dot(self.A, self.x) + self.B * u
        u2 = float(np.dot(self.K, x_next))
        r_val = float(np.dot(np.dot(x_curr.T, self.Q), x_curr) + (u**2) * self.R)

        # RLS
        if self.iter <= self.n_ls:
            idx = self.iter - 1
            self.r_ls[idx, 0] = r_val
            self.phi_ls[:, idx] = [x_curr[0,0]**2, x_curr[0,0]*x_curr[1,0], x_curr[0,0]*u, x_curr[1,0]**2, x_curr[1,0]*u, u**2]
            self.phi1_ls[:, idx] = [x_next[0,0]**2, x_next[0,0]*x_next[1,0], x_next[0,0]*u2, x_next[1,0]**2, x_next[1,0]*u2, u2**2]
            if self.iter == self.n_ls:
                diff_phi = self.phi_ls - self.phi1_ls
                self.W_H = np.dot(np.linalg.pinv(np.dot(diff_phi, diff_phi.T)), np.dot(diff_phi, self.r_ls))
        else:
            phi = np.array([x_curr[0,0]**2, x_curr[0,0]*x_curr[1,0], x_curr[0,0]*u, x_curr[1,0]**2, x_curr[1,0]*u, u**2]).reshape(-1, 1)
            phi1 = np.array([x_next[0,0]**2, x_next[0,0]*x_next[1,0], x_next[0,0]*u2, x_next[1,0]**2, x_next[1,0]*u2, u2**2]).reshape(-1, 1)
            vec_act = phi - self.gamma * phi1
            e_dt = r_val - float(np.dot(self.W_H.T, vec_act))
            
            p_scaled = (1.0 / self.f_olvido) * self.P_n
            den = (1.0/(1.0-self.f_olvido)) + float(np.dot(np.dot(vec_act.T, p_scaled), vec_act))
            L = np.dot(p_scaled, vec_act) / den
            self.W_H += L * e_dt
            self.P_n = np.dot((np.eye(6) - np.dot(L, vec_act.T)), p_scaled)
            
            W = self.W_H.flatten()
            self.h_actual = np.array([[W[0], W[1]/2, W[2]/2], [W[1]/2, W[3], W[4]/2], [W[2]/2, W[4]/2, W[5]]])

            if abs(e_dt) < self.con_error_k:
                if abs(self.h_actual[2, 2]) > 1e-4:
                    self.K = (-(1.0 / self.h_actual[2, 2]) * self.h_actual[2, 0:2]).reshape(1, 2)

        # Updates UI
        self.rect.setData(x=[0], y=[float(x_next[0,0])])
        self.pos_history.append(float(x_next[0,0]))
        self.curve_pos.setData(self.pos_history[-800:]) # Ver solo últimas 800 en tiempo real
        
        W_vals = self.W_H.flatten()
        for j in range(6):
            self.w_history[j].append(W_vals[j])
            self.w_curves[j].setData(self.w_history[j]) # Historial completo (hasta 4000)

        self.lbl_k.setText(f"K RL: {self.K[0,0]:.3f}, {self.K[0,1]:.3f}\nK OPT: {self.K_opt[0]:.3f}, {self.K_opt[1]:.3f}")
        self.lbl_iter.setText(f"Iteración: {self.iter} / {self.N}")
        self.x = x_next
        self.iter += 1

if __name__ == "__main__":
    app = QApplication(sys.argv)
    sim = MRA_W_Tracking()
    sim.show()
    sys.exit(app.exec())

/tmp/ipykernel_18620/39228761.py:125: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  u = float(np.dot(self.K, self.x)) + noise
/tmp/ipykernel_18620/39228761.py:127: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  u2 = float(np.dot(self.K, x_next))
/tmp/ipykernel_18620/39228761.py:128: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r_val = float(np.dot(np.dot(x_curr.T, self.Q), x_curr) + (u**2) * self.R)
/tmp/ipykernel_18620/39228761.py:125: DeprecationWarning: Conversi